In [1]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import requests
import pandas as pd
import json
import time
import random
import os
import re
from datetime import datetime
import datetime as dt

# ── ID Continuity ──────────────────────────────────────────────────────────
existing_max_id = 0
for csv_path in [
    "data/consumer_complaints.csv",
    "data/indiankanoon_complaints.csv",
    "data/medianama_complaints.csv",
    "data/reddit_complaints.csv",
]:
    if os.path.exists(csv_path):
        df_check = pd.read_csv(csv_path)
        ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
        if not ids.empty:
            existing_max_id = max(existing_max_id, int(ids.max()))

# Also scan txt folder directly as ultimate safety net
import re as _re
for fname in os.listdir("data/txt"):
    match = _re.search(r'NA-(\d+)\.txt', fname)
    if match:
        existing_max_id = max(existing_max_id, int(match.group(1)))

next_id = existing_max_id + 1
start_id = next_id
print(f"Starting Unique ID from: NA-{next_id:04d}")

# ── Paths & APIs ───────────────────────────────────────────────────────────
PULLPUSH_URL = "https://api.pullpush.io/reddit/search/submission/"
COMMENTS_URL = "https://api.pullpush.io/reddit/search/comment/"

TARGET_SUBREDDITS = [
    # Already existing
    "india", "LegalAdviceIndia", "IndiaInvestments",
    "bangalore", "mumbai", "delhi", "hyderabad",
    "chennai", "pune", "PersonalFinanceIndia", "scams",
    # New additions
    "kolkata",
    "IndianStockMarket",
    "digitalpaymentsindia",
    "IndiaFinance",
    "IndiaTax",
    "ahmedabad",
    "jaipur",
    "surat",
    "lucknow",
    "nagpur",
    "bhopal",
    "chandigarh",
    "kochi",
    "IndianGaming",
    "CryptoCurrencyIndia",
    "UPI",
    "Paytm",
    "PhonePe",
    "india_banking",
    "IndiaOnlineJobs",
    "indianworking",
    "recruitinghell_india",
    "AskIndia",
    "TwoXIndia",
    "indianmedstudents",
    "IndianDefense",
    "indiadiscussion",
    "CasualConversation_India",
    "developersIndia",
    "cscareerquestionsIndia",
    "IndianBanking",
    "fraudIndia",
    "cybersecurity_india",
]

OUTPUT_CSV = "data/reddit_complaints.csv"
OUTPUT_JSON = "data/reddit.json"
OUTPUT_EXCEL = "data/reddit_complaints.xlsx"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Date range for research: Jan 2014 to Dec 2025
# PullPush uses Unix timestamps
START_TIMESTAMP = int(dt.datetime(2014, 1, 1).timestamp())
END_TIMESTAMP = int(dt.datetime(2025, 12, 31).timestamp())

Starting Unique ID from: NA-27888


In [2]:
# === CELL 2: FULL KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [3]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["i lost", "i was scammed", "they took", "money deducted",
                                       "i got cheated", "duped", "fell for", "lost money",
                                       "my account", "amount debited"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["almost", "tried to scam", "i did not", "i refused",
                                         "i avoided", "beware", "warning", "how i avoided",
                                         "did not share", "suspicious"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def safe_save(df, csv_path, json_path, json_data):
    """Save via temp file to avoid PermissionError if Excel has file open."""
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        print(f"  ✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"  ⚠️ Checkpoint save failed (data still in memory): {e}")

In [4]:
# === CELL 4: PULLPUSH FETCH FUNCTION ===
def fetch_reddit_posts(keyword, subreddit, before=None, size=100):
    """Fetch posts from PullPush API for a keyword in a subreddit."""
    params = {
        "q": keyword,
        "subreddit": subreddit,
        "size": size,
        "after": START_TIMESTAMP,
        "sort": "desc",
        "sort_type": "created_utc"
    }
    if before:
        params["before"] = before

    try:
        time.sleep(random.uniform(1.0, 3.0))
        response = requests.get(PULLPUSH_URL, params=params, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            print(f"    Failed: status {response.status_code}")
            return []
        data = response.json()
        posts = data.get("data", [])
        return posts
    except Exception as e:
        print(f"    Error fetching '{keyword}' from r/{subreddit}: {e}")
        return []

In [6]:
# === CELL 5: MAIN SCRAPING LOOP ===
# Setup duplicates loading
existing_post_ids = set()
existing_reddit_urls = set()
existing_data_json = []

if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        existing_data_json = json.load(f)
    for item in existing_data_json:
        if item.get("URL"):
            existing_reddit_urls.add(item["URL"])
        if item.get("Reddit Post ID"):
            existing_post_ids.add(item["Reddit Post ID"])
        elif item.get("Notes") and "Reddit ID: " in item["Notes"]:
            existing_post_ids.add(item["Notes"].split("Reddit ID: ")[-1].split(" |")[0])

print(f"Loaded {len(existing_reddit_urls)} existing Reddit URLs to skip")
print(f"Already scraped: {len(existing_post_ids)} posts")

new_records = []
new_results_count = 0

today_date = datetime.now().strftime("%Y-%m-%d")

for subreddit in TARGET_SUBREDDITS:
    print(f"\n=== Scraping Subreddit: r/{subreddit} ===")
    
    for parent_category, subcategories in KEYWORD_TAXONOMY.items():
        for subcat, keyword_list in subcategories.items():
            for keyword in keyword_list:
                print(f"  Keyword: '{keyword}'")
                before = END_TIMESTAMP
                
                while True:
                    posts = fetch_reddit_posts(keyword, subreddit, before=before, size=100)
                    
                    if not posts:
                        break
                        
                    batch_valid = 0
                    for post in posts:
                        post_id = post.get('id', '')
                        if not post_id or post_id in existing_post_ids:
                            continue
                            
                        # Also check url to skip
                        permalink = post.get('permalink', '')
                        full_url = f"https://www.reddit.com{permalink}"
                        if full_url in existing_reddit_urls:
                            continue
                            
                        existing_post_ids.add(post_id)
                        existing_reddit_urls.add(full_url)
                            
                        created_utc = post.get('created_utc')
                        if created_utc is None:
                            continue
                        try:
                            created_utc = int(float(str(created_utc)))
                        except (ValueError, TypeError):
                            continue
                        if created_utc < START_TIMESTAMP or created_utc > END_TIMESTAMP:
                            continue
                            
                        readable_date = datetime.fromtimestamp(created_utc).strftime("%Y-%m-%d")
                        
                        selftext = post.get('selftext', '').strip()

                        # Handle deleted, removed, or empty posts
                        if not selftext or selftext in ['[deleted]', '[removed]']:
                            post_body = f"[No body text — title only]\n\nTitle contains full narrative:\n{post.get('title', '')}"
                        else:
                            post_body = clean_text(selftext)
                            
                        title = post.get('title', '')
                        score = post.get('score', 0)
                        author = post.get('author', 'unknown')
                        
                        # Generate next id
                        global next_id
                        assigned_id = f"NA-{next_id:04d}"
                        txt_filename = f"{assigned_id}.txt"
                        next_id += 1
                        
                        # Build txt file content
                        txt_content = f"SUBREDDIT: r/{post.get('subreddit', '')}\n"
                        txt_content += f"TITLE: {post.get('title', '')}\n"
                        txt_content += f"DATE: {readable_date}\n"
                        txt_content += f"SCORE: {post.get('score', 0)}\n"
                        txt_content += f"URL: {full_url}\n\n"
                        txt_content += "--- POST TEXT ---\n\n"
                        txt_content += post_body

                        # Use title + post body combined for narrative classification
                        full_content = post.get('title', '') + ' ' + post_body
                        narrative_type = classify_narrative_type(full_content)
                        
                        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_txt:
                            f_txt.write(txt_content)
                            
                        record = {
                            "Reddit Post ID": post_id,
                            "Unique ID": assigned_id,
                            "Date of Collection": today_date,
                            "Collector Name": "Soubhik Sarkar",
                            "Source Platform": "Reddit",
                            "Source Publication": f"r/{post.get('subreddit', '')}",
                            "Original Date": readable_date,
                            "Title/Headline": title,
                            "URL": full_url,
                            "Search Query Used": keyword,
                            "Fraud Category": parent_category,
                            "Fraud Subcategory": subcat,
                            "Narrative Type": narrative_type,
                            "TXT File Name": txt_filename,
                            "Notes": f"Author: {author} | Upvotes: {score} | Reddit ID: {post_id}"
                        }
                        
                        raw_item = {
                            "Reddit Post ID": post_id,
                            "URL": full_url,
                            "Title/Headline": title,
                            "Original Date": readable_date,
                            "Author": author,
                            "StructuredData": record
                        }

                        new_records.append(record)
                        existing_data_json.append(raw_item)
                        
                        new_results_count += 1
                        batch_valid += 1
                        
                    if len(posts) > 0:
                        before = posts[-1].get('created_utc')
                    else:
                        break
                        
                    print(f"    Added {batch_valid} posts, moving 'before' to {before}")
                    
                    if new_results_count % 50 == 0 and new_results_count > 0:
                        df_temp = pd.DataFrame([x["StructuredData"] for x in existing_data_json])
                        safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, existing_data_json)

print(f"\nScraping phase complete.")

Loaded 12202 existing Reddit URLs to skip
Already scraped: 12202 posts

=== Scraping Subreddit: r/india ===
  Keyword: 'cyber crime'
    Added 0 posts, moving 'before' to 1707421112.0
    Added 0 posts, moving 'before' to 1635571953
    Added 0 posts, moving 'before' to 1496600764
    Added 42 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'before' to 1390906913
    Added 0 posts, moving 'befo

KeyboardInterrupt: 

In [ ]:
# === CELL 6: FINAL SAVE BEFORE CLEANUP ===
if new_results_count > 0:
    df_new = pd.DataFrame(new_records)
    
    if os.path.exists(OUTPUT_CSV):
        df_existing = pd.read_csv(OUTPUT_CSV)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
        # Deduplicate
        df_combined.drop_duplicates(subset=["URL"], keep="last", inplace=True)
    else:
        df_combined = df_new

    # Save CSV and JSON
    safe_save(df_combined, OUTPUT_CSV, OUTPUT_JSON, existing_data_json)
    
    try:
        df_combined.to_excel(OUTPUT_EXCEL, index=False)
        print(f"  ✅ Saved Excel to {OUTPUT_EXCEL}")
    except Exception as e:
        print(f"  ⚠️ Could not save Excel: {e}")
        
    print(f"✅ Final save complete. Total records: {len(df_combined)}")
    display(df_combined.tail(3))
else:
    df_combined = pd.read_csv(OUTPUT_CSV) if os.path.exists(OUTPUT_CSV) else pd.DataFrame()
    print("No new data to save.")

  ✅ Checkpoint saved: 57118 records
  ✅ Saved Excel to data/reddit_complaints.xlsx
✅ Final save complete. Total records: 57118


,Unique ID,Date of Collection,Collector Name,Source Platform,Source Publication,Original Date,Title/Headline,URL,Search Query Used,Fraud Category,Fraud Subcategory,Narrative Type,TXT File Name,Notes
57115,NA-72501,2026-04-01,Soubhik Sarkar,Reddit,r/Scams,2020-08-16,WHVPOID,https://www.reddit.com/r/Scams/comments/iald4c...,online harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,NEAR-MISS,NA-72501.txt,Reddit ID: iald4c | Score: 0 | Author: RayRay5454
57116,NA-72502,2026-04-01,Soubhik Sarkar,Reddit,r/Scams,2018-03-15,Received a call from a paramedic claiming a fe...,https://www.reddit.com/r/Scams/comments/84nead...,online harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,NEAR-MISS,NA-72502.txt,Reddit ID: 84nead | Score: 114 | Author: thisi...
57117,NA-72503,2026-04-01,Soubhik Sarkar,Reddit,r/Scams,2024-09-19,Pearl Organisation Dehradun is totally scam on...,https://www.reddit.com/r/Scams/comments/1fkl50...,digital harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,THIRD-PARTY,NA-72503.txt,Reddit ID: 1fkl509 | Score: 0 | Author: Chemic...


In [ ]:
# === CELL 7: DIAGNOSTIC TEST CELL ===
# Test one keyword in one subreddit
test_posts = fetch_reddit_posts("UPI fraud", "india", size=3)
print(f"Found {len(test_posts)} posts")
if test_posts:
    p = test_posts[0]
    print(f"Title: {p.get('title')}")
    print(f"Date: {datetime.fromtimestamp(p['created_utc']).strftime('%Y-%m-%d')}")
    print(f"Subreddit: r/{p.get('subreddit')}")
    print(f"Text preview: {p.get('selftext', '')[:300]}")
    print(f"URL: https://www.reddit.com{p.get('permalink')}")

Found 3 posts
Title: Zepto Delivery Agent Misused OTP — Product Not Delivered, ₹500 Free Cash Lost — Beware!
Date: 2025-04-28
Subreddit: r/india
Text preview: Sharing a serious incident that happened recently with Zepto, so others don’t fall for the same trap.

I ordered a **boAt Wave Aura Smartwatch** on Zepto for just **₹490**, using my **₹500 Free Cash**. Normally, this smartwatch costs around **₹900**, so it was a great deal. I chose **Cash on Deliver
URL: https://www.reddit.com/r/india/comments/1k9tgsj/zepto_delivery_agent_misused_otp_product_not/


In [ ]:
# === CELL 8: SCAMS SUBREDDIT POST-CLEANUP ===
import pandas as pd
import os
import re
import json

print("Starting Cleanup Process...")

# Only run if we have data files
if os.path.exists(OUTPUT_CSV) and os.path.exists(OUTPUT_JSON):
    df = pd.read_csv(OUTPUT_CSV)
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        existing_data_json = json.load(f)

    # Create case-insensitive regex for India keywords
    india_keywords = [
        "india", "indian", "rupee", "inr", "mumbai", "delhi", "bangalore", 
        "bengaluru", "hyderabad", "chennai", "kolkata", "pune", "noida", 
        "gurgaon", "ahmedabad", "upi", "paytm", "phonepe", "gpay", "bhim", 
        "aadhaar", "pan card", "rbi", "sbi", "hdfc", "icici", "kotak", 
        "axis bank", "cyber dost", "cybercrime.gov.in"
    ]
    keyword_pattern = re.compile(r'\b(?:' + '|'.join(india_keywords) + r')\b', re.IGNORECASE)

    print(f"Total records before cleanup: {len(df)}")

    # 1. Drop duplicates based on Title/Headline, URL, and TXT File Name
    print("\nStep 1: Removing duplicates...")
    len_before_dedups = len(df)
    
    # We will identify which txt files to delete too
    txts_before = set(df["TXT File Name"].dropna())
    
    # Deduplicate by Title/Headline
    df_dedup = df.drop_duplicates(subset=["Title/Headline"], keep="first")
    # Deduplicate by URL
    df_dedup = df_dedup.drop_duplicates(subset=["URL"], keep="first")
    # Deduplicate by TXT File Name
    df_dedup = df_dedup.drop_duplicates(subset=["TXT File Name"], keep="first")
    
    txts_after = set(df_dedup["TXT File Name"].dropna())
    txts_to_delete = txts_before - txts_after
    
    duplicates_removed = len_before_dedups - len(df_dedup)
    print(f"Duplicate records removed: {duplicates_removed}")
    
    # Delete the orphan TXT files left by duplicates
    deleted_txt_count = 0
    for t_file in txts_to_delete:
        t_path = os.path.join(TXT_DIR, str(t_file))
        if os.path.exists(t_path):
            os.remove(t_path)
            deleted_txt_count += 1
    print(f"TXT files deleted (duplicates): {deleted_txt_count}")

    # 2. Filter r/scams
    print("\nStep 2: Filtering non-India posts from r/Scams...")
    
    def is_india_relevant(row):
        # Only scrub r/Scams. Other subreddits are assumed relevant by default.
        if str(row.get("Source Publication")).lower() != "r/scams":
            return True
            
        title = str(row.get("Title/Headline", ""))
        if keyword_pattern.search(title):
            return True
            
        txt_path = os.path.join(TXT_DIR, str(row.get("TXT File Name", "")))
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                if keyword_pattern.search(content):
                    return True
        return False

    indices_to_drop = []
    scams_txts_to_delete = []
    
    for idx, row in df_dedup.iterrows():
        if not is_india_relevant(row):
            indices_to_drop.append(idx)
            if pd.notna(row.get("TXT File Name")):
                scams_txts_to_delete.append(str(row.get("TXT File Name")))
            
    df_final = df_dedup.drop(indices_to_drop)
    removed_count = len(indices_to_drop)
    
    for t_file in scams_txts_to_delete:
        t_path = os.path.join(TXT_DIR, t_file)
        if os.path.exists(t_path):
            os.remove(t_path)
            
    print(f"Removed {removed_count} r/scams posts lacking India context.")

    # 3. Synchronize JSON and save files
    print("\nStep 3: Saving cleaned dataset correctly...")
    valid_urls = set(df_final["URL"].dropna())
    cleaned_json = [item for item in existing_data_json if item.get("URL") in valid_urls]
    
    # Overwrite CSV directly
    df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print("✅ CSV updated.")

    try:
        df_final.to_excel(OUTPUT_EXCEL, index=False)
        print(f"✅ Saved Excel to {OUTPUT_EXCEL}")
    except Exception as e:
        print(f"⚠️ Could not save Excel: {e}")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(cleaned_json, f, indent=4, ensure_ascii=False)
    print("✅ JSON updated.")

    print(f"\n✅ CLEANUP COMPLETE!")
    print(f"- Final record count in CSV: {len(df_final)}")
    print(f"- Final record count in JSON: {len(cleaned_json)}")
    
    # Update global dataframe for the next summary cell
    df_combined = df_final
else:
    print("⚠️ Required data files not found for cleanup.")

Step 1: Filtering non-India posts from r/scams...
Step 2: Removing duplicates...
Step 3: Deleting orphaned TXT files...

Step 4: Cleanup Summary
  - Total records before cleanup: 56818
  - Records removed from r/scams (non-India): 44616
  - Duplicate records removed: 0
  - Final record count: 12202
  - TXT files deleted: 44616

  - Breakdown by Subreddit:
Source Publication
r/india                   5318
r/LegalAdviceIndia        2282
r/Scams                   1056
r/bangalore                944
r/delhi                    853
r/hyderabad                466
r/mumbai                   329
r/personalfinanceindia     284
r/Chennai                  247
r/pune                     223
r/IndiaInvestments         200

Step 5: Saving cleaned data safely...
  ⚠️ Could not save Excel: No engine for filetype: 'tmp'
  - JSON reduced from 56818 to 12202 items.

✅ Cleanup fully complete!


In [ ]:
# === CELL 9: SUMMARY ===
print(f"New records this session: {new_results_count}")
print(f"Total Reddit records now: {len(df_combined)}")
print(f"New IDs: NA-{start_id:04d} to NA-{next_id-1:04d}")
print("\n--- Source Publication Counts ---")
print(df_combined["Source Publication"].value_counts())

data/consumer_complaints.csv: max ID = NA-4318
data/indiankanoon_complaints.csv: max ID = NA-13176
data/reddit_complaints.csv: max ID = NA-72465

True highest ID across everything: NA-72465
Reddit records to renumber: 12202
New IDs will run: NA-72466 to NA-84667
TXT files renamed: 12202
TXT files skipped (already correct): 0
TXT files not found: 0

✅ Done! Reddit IDs: NA-72466 to NA-84667


In [15]:
import pandas as pd
import os

for csv_path in [
    "data/consumer_complaints.csv",
    "data/indiankanoon_complaints.csv", 
    "data/reddit_complaints.csv",
    "data/medianama_complaints.csv"
]:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        ids = df["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
        if not ids.empty:
            print(f"{os.path.basename(csv_path)}: {len(df)} records | IDs: NA-{int(ids.min()):04d} to NA-{int(ids.max()):04d}")
        else:
            print(f"{os.path.basename(csv_path)}: {len(df)} records | No IDs found")
    else:
        print(f"NOT FOUND: {csv_path}")

consumer_complaints.csv: 4318 records | IDs: NA-0001 to NA-4318
indiankanoon_complaints.csv: 8858 records | IDs: NA-4319 to NA-13176
reddit_complaints.csv: 12202 records | IDs: NA-72466 to NA-84667
medianama_complaints.csv: 2509 records | IDs: NA-13177 to NA-15685


In [ ]:
import pandas as pd
import os
import json
import re

# Reddit should start right after Medianama's last ID
start_id = 15686  # NA-15685 was last Medianama record

df = pd.read_csv("data/reddit_complaints.csv")
print(f"Reddit records: {len(df)}")
print(f"New IDs will be: NA-{start_id:04d} to NA-{start_id + len(df) - 1:04d}")

new_ids = [f"NA-{start_id + i:04d}" for i in range(len(df))]

# Phase 1 — rename all current txt files to temp names first
print("\nPhase 1: Renaming to temp names...")
temp_map = {}
for i, row in df.iterrows():
    old_txt = str(row.get("TXT File Name", ""))
    old_path = os.path.join("data/txt", old_txt)
    if os.path.exists(old_path):
        temp_name = f"_TEMP_{i}_.txt"
        temp_path = os.path.join("data/txt", temp_name)
        os.rename(old_path, temp_path)
        temp_map[i] = temp_name

print(f"Renamed {len(temp_map)} files to temp names")

# Phase 2 — rename temp files to correct new names
print("Phase 2: Renaming to correct new IDs...")
renamed = 0
for i, row in df.iterrows():
    if i not in temp_map:
        continue
    temp_path = os.path.join("data/txt", temp_map[i])
    new_txt = f"NA-{start_id + i - df.index[0]:04d}.txt"
    new_path = os.path.join("data/txt", new_txt)
    if os.path.exists(temp_path):
        os.rename(temp_path, new_path)
        renamed += 1
    df.at[i, "TXT File Name"] = new_txt

df["Unique ID"] = new_ids

print(f"Renamed {renamed} txt files")

# Save CSV
temp_csv = "data/reddit_complaints.csv.tmp"
df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
os.replace(temp_csv, "data/reddit_complaints.csv")
df.to_excel("data/reddit_complaints.xlsx", index=False)
print("CSV and Excel saved")

# Update JSON
if os.path.exists("data/reddit.json"):
    with open("data/reddit.json", "r", encoding="utf-8") as f:
        json_data = json.load(f)

    url_to_new_id = dict(zip(df["URL"], df["Unique ID"]))
    url_to_new_txt = dict(zip(df["URL"], df["TXT File Name"]))

    for item in json_data:
        url = item.get("URL", "")
        if url in url_to_new_id:
            item["Unique ID"] = url_to_new_id[url]
            item["TXT File Name"] = url_to_new_txt[url]
            if "StructuredData" in item:
                item["StructuredData"]["Unique ID"] = url_to_new_id[url]
                item["StructuredData"]["TXT File Name"] = url_to_new_txt[url]

    with open("data/reddit.json", "w", encoding="utf-8") as f:
        json.dump(json_data, f, indent=4, ensure_ascii=False)
    print("JSON updated")

# Verify
df_check = pd.read_csv("data/reddit_complaints.csv")
ids = df_check["Unique ID"].dropna().str.extract(r'(\d+)')[0].astype(float)
print(f"\n✅ Done! Reddit IDs: NA-{int(ids.min()):04d} to NA-{int(ids.max()):04d}")
print(f"Total Reddit records: {len(df_check)}")

Reddit records: 12202
New IDs will be: NA-15686 to NA-27887

Phase 1: Renaming to temp names...
Renamed 12202 files to temp names
Phase 2: Renaming to correct new IDs...
Renamed 12202 txt files
CSV and Excel saved
JSON updated

✅ Done! Reddit IDs: NA-15686 to NA-27887
Total Reddit records: 12202


: 